In [1]:
import os
import cv2
import torch
import numpy as np
from torch.utils.data import Dataset
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm import tqdm
import os
import json
from datetime import datetime

from degrade import SRDataset
from model import DegradationAwareSR

In [2]:

def _apply_gaussian_blur(img, rng):
    k = rng.choice([3, 5, 7])  # Reduce kernel sizes (faster)
    sigma = rng.uniform(0.5, 2.0)  # Reduce sigma range
    return cv2.GaussianBlur(img, (k, k), sigmaX=sigma, sigmaY=sigma)

def _apply_noise(img, rng):
    """Faster noise: only gaussian"""
    sigma = rng.uniform(2.0, 8.0)
    noise = rng.normal(0, sigma, img.shape).astype(np.float32)
    return np.clip(img + noise, 0, 255)

def _apply_downsample(img, scale, rng):
    """Single interpolation method (fastest)"""
    h, w = img.shape[:2]
    # Use INTER_AREA for downsampling (built-in optimization)
    return cv2.resize(img, (w // scale, h // scale), interpolation=cv2.INTER_AREA)

def apply_lr1_degradation(img, scale, rng):
    """LR1: Blur + Noise (no JPEG) - FAST"""
    img = img.astype(np.float32)
    img = _apply_gaussian_blur(img, rng)
    img = _apply_downsample(img, scale, rng)
    img = _apply_noise(img, rng)
    return np.clip(img, 0, 255).astype(np.uint8)

def apply_lr2_degradation(img, scale, rng):
    """LR2: Gaussian blur + downsample (no JPEG) - FASTER"""
    img = img.astype(np.float32)
    img = _apply_gaussian_blur(img, rng)  # Different blur params
    img = _apply_downsample(img, scale, rng)
    return np.clip(img, 0, 255).astype(np.uint8)


In [3]:

class SRDataset(Dataset):
    def __init__(self, hr_dir, lr_dir=None, scale=4, degrade=True, crop_size=128, 
                 return_lr_pair=False, num_workers=0, cache_size=500):
        self.hr_dir = hr_dir
        self.scale = scale
        self.crop_size = crop_size
        self.return_lr_pair = return_lr_pair
        
        # Get image list ONCE - Windows compatible
        valid_extensions = ('.png', '.jpg', '.jpeg', '.JPG', '.JPEG', '.PNG')
        self.images = []
        
        for f in os.listdir(hr_dir):
            if f.lower().endswith(valid_extensions):
                self.images.append(f)
        
        self.images = sorted(self.images)  # Deterministic order
        
        # Initialize RNG per worker
        self.rng = np.random.RandomState()
        
        # Pre-compute LR pairs on initialization (one-time cost)
        self.cache = {}
        self._precompute_degradations()
    
    def _precompute_degradations(self):
        """Pre-degrade all images once (happens at init, not per epoch)"""
        print(f"🔄 Pre-computing degradations for {len(self.images)} images...")
        
        failed_count = 0
        for idx, name in enumerate(self.images):
            hr_path = os.path.join(self.hr_dir, name)
            
            try:
                hr = cv2.imread(hr_path)
                
                if hr is None:
                    print(f"  ⚠️ Failed to load: {name}")
                    failed_count += 1
                    continue
                
                # Store HR in cache
                self.cache[name] = {
                    'hr': hr,
                    'lr1': apply_lr1_degradation(hr.copy(), self.scale, self.rng),
                    'lr2': apply_lr2_degradation(hr.copy(), self.scale, self.rng),
                }
                
                if (idx + 1) % 500 == 0:
                    print(f"  {idx+1}/{len(self.images)} images cached")
            
            except Exception as e:
                print(f"  ⚠️ Error processing {name}: {e}")
                failed_count += 1
                continue
        
        print(f"✅ Cache complete: {len(self.cache)} images (failed: {failed_count})")
        
        # Update image list to only valid cached images
        self.images = list(self.cache.keys())

    def __len__(self):
        return len(self.cache)

    def _random_crop(self, img):
        if self.crop_size is None:
            return img
        h, w = img.shape[:2]
        if h < self.crop_size or w < self.crop_size:
            return img
        top = self.rng.randint(0, max(1, h - self.crop_size + 1))
        left = self.rng.randint(0, max(1, w - self.crop_size + 1))
        return img[top:top + self.crop_size, left:left + self.crop_size]

    def __getitem__(self, idx):
        name = self.images[idx]
        
        # All data is already cached (instant retrieval)
        data = self.cache[name]
        hr = data['hr'].copy()
        lr1 = data['lr1'].copy()
        lr2 = data['lr2'].copy()
        
        # # Random crop (per epoch variation)
        # hr = self._random_crop(hr)
        # lr1 = self._random_crop(lr1)
        # lr2 = self._random_crop(lr2)
        
        
        
        # Define sizes
        lr_size = self.crop_size
        hr_size = lr_size * self.scale

        h, w = hr.shape[:2]

        if h < hr_size or w < hr_size:
            hr = cv2.resize(hr, (hr_size, hr_size))
            lr1 = cv2.resize(lr1, (lr_size, lr_size))
            lr2 = cv2.resize(lr2, (lr_size, lr_size))
        else:
            top = self.rng.randint(0, h - hr_size + 1)
            left = self.rng.randint(0, w - hr_size + 1)

            # HR crop
            hr = hr[top:top+hr_size, left:left+hr_size]

            # LR crop (aligned)
            lr_top = top // self.scale
            lr_left = left // self.scale

            lr1 = lr1[lr_top:lr_top+lr_size, lr_left:lr_left+lr_size]
            lr2 = lr2[lr_top:lr_top+lr_size, lr_left:lr_left+lr_size]
        
        # BGR → RGB (fast)
        hr = cv2.cvtColor(hr, cv2.COLOR_BGR2RGB)
        lr1 = cv2.cvtColor(lr1, cv2.COLOR_BGR2RGB)
        lr2 = cv2.cvtColor(lr2, cv2.COLOR_BGR2RGB)
        
        # To tensor (fast)
        hr = torch.from_numpy(hr).float() / 255.0
        lr1 = torch.from_numpy(lr1).float() / 255.0
        lr2 = torch.from_numpy(lr2).float() / 255.0
        
        # HWC → CHW
        hr = hr.permute(2, 0, 1)
        lr1 = lr1.permute(2, 0, 1)
        lr2 = lr2.permute(2, 0, 1)
        
        return lr1, lr2, hr

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ResidualBlock(nn.Module):
    """Minimal residual block"""
    def __init__(self, channels=32):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        return x + self.conv2(self.relu(self.conv1(x)))


In [5]:


class DegradationAwareSR(nn.Module):
    """
    Fast degradation-aware SR.
    
    Key changes:
    - Smaller feature channels (32 instead of 64)
    - Fewer residual blocks (4 instead of 8)
    - Single degradation encoder (shared)
    - No complex modulation
    """
    
    def __init__(self, scale=4, d_channels=8, feat_channels=32):
        super().__init__()
        self.scale = scale
        
        # Lightweight degradation encoder
        self.deg_enc = nn.Sequential(
            nn.Conv2d(3, feat_channels, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(feat_channels, feat_channels, 3, padding=1),
            nn.ReLU(inplace=True),
        )
        
        # Degradation heads
        self.deg_head1 = nn.Conv2d(feat_channels, d_channels, 3, padding=1)
        self.deg_head2 = nn.Conv2d(feat_channels, d_channels, 3, padding=1)
        
        # Simple modulation
        self.mod = nn.Conv2d(d_channels, 3, 1)
        
        # SR pathway (lightweight)
        self.head = nn.Conv2d(3, 32, 3, padding=1)
        self.body = nn.Sequential(*[ResidualBlock(32) for _ in range(4)])  # 4 blocks instead of 8
        self.tail = nn.Conv2d(32, 3, 3, padding=1)

    def forward(self, lr1, lr2=None):
        # Shared degradation encoding
        f1 = self.deg_enc(lr1)
        d1 = torch.sigmoid(self.deg_head1(f1))
        
        if lr2 is not None:
            f2 = self.deg_enc(lr2)
            d2 = torch.sigmoid(self.deg_head2(f2))
            d_fused = 0.5 * (d1 + d2)
        else:
            d2 = None
            d_fused = d1
        
        # Simple modulation
        mod = torch.tanh(self.mod(d_fused)) * 0.2
        x = lr1 + mod
        x = torch.clamp(x, 0, 1)
        
        # Upsampling
        x = F.interpolate(x, scale_factor=self.scale, mode="bicubic", align_corners=False)
        
        # SR network
        x = self.head(x)
        x = self.body(x)
        x = self.tail(x)
        
        # Skip connection
        skip = F.interpolate(lr1, scale_factor=self.scale, mode="bicubic", align_corners=False)
        sr = x + skip
        sr = torch.clamp(sr, 0, 1)
        
        if lr2 is not None:
            return sr, d1, d2, d_fused
        else:
            return sr, d_fused

In [6]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
EPOCHS = 30
BATCH_SIZE = 32  # INCREASED from 8 → 32 (faster convergence)
NUM_WORKERS = 0  # WINDOWS: Must be 0 (multiprocessing issues on Windows)
PIN_MEMORY = False  # WINDOWS: Disable for compatibility
LR = 2e-4  # INCREASED learning rate (faster convergence)
CONSIST_WEIGHT = 0.1  # REDUCED (focus on SR quality)
CROP_SIZE = 96  # REDUCED from 128 (faster processing)
SAVE_PATH = "checkpoints"

os.makedirs(SAVE_PATH, exist_ok=True)

In [7]:


print(f"Device: {DEVICE}")
print("Loading dataset...")
train_dataset = SRDataset(
    "train/HR",
    scale=4,
    degrade=True,
    crop_size=CROP_SIZE,
    return_lr_pair=True,
)

train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True,
    num_workers=0,
    pin_memory=False, 
)


Device: cuda
Loading dataset...
🔄 Pre-computing degradations for 3450 images...
  500/3450 images cached
  1000/3450 images cached
  1500/3450 images cached
  2000/3450 images cached
  2500/3450 images cached
  3000/3450 images cached
✅ Cache complete: 3450 images (failed: 0)


In [8]:

# ==============================================================================
# MODEL & OPTIMIZATION
# ==============================================================================

model = DegradationAwareSR(scale=4, d_channels=8, feat_channels=32).to(DEVICE)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,} (smaller = faster)")

criterion = nn.L1Loss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

# Aggressive scheduler (decay faster)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.7)


Model parameters: 90,542 (smaller = faster)


In [12]:
def psnr(sr, hr):
    sr = torch.clamp(sr, 0, 1)
    hr = torch.clamp(hr, 0, 1)
    mse = torch.mean((sr - hr) ** 2)
    if mse == 0:
        return torch.tensor(100.0)
    return 20 * torch.log10(1.0 / torch.sqrt(mse))


In [13]:
history = {"epoch": [], "loss": [], "loss_sr": [], "loss_consist": []}
best_epoch = 0

print(f"TRAINING CONFIG")
print(f"{'='*70}")
print(f"Batch Size:        {BATCH_SIZE}")
print(f"Crop Size:         {CROP_SIZE}")
print(f"Learning Rate:     {LR}")
print(f"Consistency Weight: {CONSIST_WEIGHT}")
print(f"Total Params:      {total_params:,}")
print(f"Device:            {DEVICE}")
print(f"{'='*70}\n")


TRAINING CONFIG
Batch Size:        32
Crop Size:         96
Learning Rate:     0.0002
Consistency Weight: 0.1
Total Params:      90,542
Device:            cuda



In [15]:

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0
    train_loss_sr = 0
    train_loss_consist = 0

    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")

    for lr1, lr2, hr_img in loop:
        lr1 = lr1.to(DEVICE)
        lr2 = lr2.to(DEVICE)
        hr_img = hr_img.to(DEVICE)
        
        # Forward
        sr, d1, d2, _ = model(lr1, lr2)
        
        # print(lr1.shape, sr.shape, hr_img.shape)

        # Losses
        loss_sr = criterion(sr, hr_img)
        loss_consist = criterion(d1, d2)
        loss = loss_sr + CONSIST_WEIGHT * loss_consist

        # Backward
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        # Accumulate
        train_loss += loss.item()
        train_loss_sr += loss_sr.item()
        train_loss_consist += loss_consist.item()

        loop.set_postfix(
            loss=loss.item(),
            lr_sr=loss_sr.item(),
            lr_cs=loss_consist.item()
        )

    # Average losses
    n_batches = len(train_loader)
    train_loss /= n_batches
    train_loss_sr /= n_batches
    train_loss_consist /= n_batches

    # Logging
    history["epoch"].append(epoch + 1)
    history["loss"].append(train_loss)
    history["loss_sr"].append(train_loss_sr)
    history["loss_consist"].append(train_loss_consist)

    print(f"\n{'='*70}")
    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"  Loss Total:      {train_loss:.6f}")
    print(f"  Loss SR:         {train_loss_sr:.6f}")
    print(f"  Loss Consistency:{train_loss_consist:.6f}")
    print(f"  LR:              {optimizer.param_groups[0]['lr']:.2e}")
    print(f"{'='*70}\n")

    # Save checkpoint every epoch
    torch.save(model.state_dict(), os.path.join(SAVE_PATH, f"model_epoch_{epoch+1}.pth"))

    # Save best
    if epoch == 0 or train_loss < min(history["loss"][:-1]):
        best_epoch = epoch + 1
        torch.save(model.state_dict(), os.path.join(SAVE_PATH, "model_best.pth"))
        print(f"✅ Best model saved at epoch {best_epoch}\n")

    scheduler.step()

print(f"\n{'='*70}")
print(f"✅ TRAINING COMPLETE")
print(f"Best epoch: {best_epoch}")
print(f"Total training time: ~{EPOCHS * 0.1} hours (approx)")

Epoch 1/30:   0%|          | 0/108 [00:02<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 576.00 MiB. GPU 0 has a total capacity of 4.00 GiB of which 0 bytes is free. Of the allocated memory 10.43 GiB is allocated by PyTorch, and 56.77 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)